# A–B · Audit baseline và kiểm chứng direct retrieval

**Mục tiêu:** giữ nguyên snapshot 24 caption, kiểm tra 40 ứng viên đã khóa và chọn các query direct có ảnh dương được người chấm xác minh. `source_group` là nhãn bundle, không phải mức suy luận đã kiểm chứng. Ảnh âm chỉ là ứng viên tham khảo. Notebook không tái tạo hoặc ghi đè workbook đã điền.


In [ ]:
from pathlib import Path
import hashlib, sys
import pandas as pd
ROOT = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive/graphml-project3').is_dir() else next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'graphml-project3').is_dir())
DUONG = ROOT/'graphml-project3/duong'
sys.path.insert(0,str(DUONG))
from xai.review_workbook import baseline_review_pack, create_review_workbook
from xai.image_download import fetch_review_image
from xai.query_registry import validated_registry_rows, import_reviewed_queries, read_case_review_workbook, direct_eligibility_reason
SNAP = DUONG/'runs/qa_vg_psg_xai/xai-20261121-130842'
REG = DUONG/'registry/query_candidates_frozen.csv'
assert SNAP.is_dir() and REG.is_file()
print('Frozen registry SHA256:',hashlib.sha256(REG.read_bytes()).hexdigest())


## 1 · Audit snapshot cũ

Kiểm tra 216 dòng, 12 query/bộ × 3 seed × 3 mức k. Đây là leave-one-edge-out; path validity và evidence được báo riêng. Mẫu số sparsity cũ chỉ là vùng sàng lọc.

In [11]:
old=pd.read_csv(SNAP/'analysis/xai_results_corrected.csv')
assert len(old)==216 and old.groupby('dataset').caption_id.nunique().to_dict()=={'vg':12,'psg':12}
display(old[old.requested_k==3].groupby('dataset').agg(n=('caption_id','size'),query_count=('caption_id','nunique'),screen_coverage=('screen_coverage','mean'),final_drop=('final_score_drop','mean'),random_drop=('random_final_score_drop_mean','mean'),evidence_known=('evidence_known_rate','mean')))


,n,query_count,screen_coverage,final_drop,random_drop,evidence_known
dataset,,,,,,
psg,36,12,0.004774,1.722630,0.726198,0.0
vg,36,12,0.008687,0.901547,0.172638,0.0


## 2 · Cổng query/ảnh direct

40 ứng viên được chọn trước khi xem XAI. Chỉ `verified_group=direct`, `gt_relevance=supported` và `review_status=approved` với người chấm rõ ràng mới vào lượt mới; cần ít nhất 10 query mỗi dataset. Ảnh âm khác quan hệ không còn là điều kiện của thí nghiệm này. Giữ mọi nhãn nguồn và kết quả không đủ điều kiện để báo cáo attrition.


In [ ]:
registry=pd.read_csv(REG,keep_default_na=False)
assert len(registry)==40
display(registry.groupby(['dataset','source_group']).size())
review_path=DUONG/'review/query_candidates_40.xlsx'
review_rows=read_case_review_workbook(review_path)
review=pd.DataFrame(review_rows)
display(review.groupby(['dataset','verified_group','review_status'],dropna=False).size())
review['direct_gt_candidate']=(review.verified_group=='direct') & (review.gt_relevance=='supported')
review['eligibility_reason']=review.apply(direct_eligibility_reason,axis=1)
display(review.groupby(['dataset','eligibility_reason']).size())
display(review.groupby('dataset').agg(direct_gt_candidates=('direct_gt_candidate','sum'),approved=('review_status',lambda x:(x=='approved').sum())))
print('Direct with GT supported:',review.direct_gt_candidate.sum(),'/ 40')


## 3 · Xuất registry đã duyệt

`query_candidates_40.xlsx` chứa 40 nhãn đã duyệt. Notebook xuất registry đầy đủ và audit lý do chọn/loại; 34 query direct có GT `supported` đi tiếp. Nếu sửa nhãn, chạy lại cell này để làm mới CSV.


In [ ]:
REVIEWED_WORKBOOK=DUONG/'review/query_candidates_40.xlsx'
assert REVIEWED_WORKBOOK.is_file(), REVIEWED_WORKBOOK
EXPORT_DIRECT_REGISTRY=True  # workbook chuẩn đã được duyệt
if EXPORT_DIRECT_REGISTRY:
    verified=import_reviewed_queries(REG,REVIEWED_WORKBOOK,DUONG/'registry/queries_direct_verified.csv')
    audit=pd.read_csv(verified,keep_default_na=False)
    audit['eligibility_reason']=audit.apply(direct_eligibility_reason,axis=1)
    audit.to_csv(DUONG/'registry/direct_query_eligibility_audit.csv',index=False)
    selected=validated_registry_rows(verified)
    print('Approved direct queries:',len(selected))
    display(pd.DataFrame(selected).groupby('dataset').size())
else:
    print('Workbook hiện tại được giữ nguyên. Chưa xuất registry đã duyệt.')
